# 🟠 Kafka — Parte 3: Streaming com Transformações, Janelas, Agregações e Alertas

---

> **Pré-requisito:** Kafka rodando (Parte 1). kafka-python instalado (Parte 2).

> **Objetivo:** Construir pipelines de streaming que transformam eventos em tempo real, calculam agregações em janelas de tempo e disparam alertas baseados em regras.

| Etapa | O que faremos |
|---|---|
| **1. Setup** | Tópicos de entrada, saída e alertas |
| **2. Transformação** | Enriquecer e filtrar eventos em stream |
| **3. Janelas Tumbling** | Agregar por janela fixa de tempo |
| **4. Janelas Sliding** | Agregar com janela deslizante |
| **5. Alertas** | Disparar alertas baseados em regras |

In [ ]:
# ============================================================
# ETAPA 1 — SETUP KAFKA (execute sempre que abrir o notebook)
# ============================================================
import subprocess, os, time, logging, sys

logging.getLogger("kafka").setLevel(logging.CRITICAL)

KAFKA_VERSION = "3.7.0"
SCALA_VERSION = "2.13"
KAFKA_DIR     = f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_URL     = f"https://archive.apache.org/dist/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
KAFKA_ARCHIVE = "/tmp/kafka.tgz"
BIN           = f"{KAFKA_DIR}/bin"

def porta_aberta(porta):
    return subprocess.run(
        ["nc", "-z", "-w1", "localhost", str(porta)],
        capture_output=True
    ).returncode == 0

def aguardar_porta(porta, label, max_seg=35):
    print(f"   Aguardando {label}", end="")
    for _ in range(max_seg):
        time.sleep(1)
        print(".", end="", flush=True)
        if porta_aberta(porta):
            print(" OK")
            return True
    print(" ERROR")
    return False

# -- 1. Java --
print("-- 1/6  Java --")
if subprocess.run(["java", "-version"], capture_output=True).returncode != 0:
    print("   Instalando Java...")
    os.system("apt-get install -y -q default-jre-headless 2>/dev/null")
print("   Java OK")

# -- 2. Netcat --
print("-- 2/6  Netcat --")
if subprocess.run(["which", "nc"], capture_output=True).returncode != 0:
    print("   Instalando netcat...")
    os.system("apt-get install -y -q netcat-openbsd 2>/dev/null")
print("   Netcat OK")

# -- 3. kafka-python-ng --
print("-- 3/6  kafka-python-ng --")
try:
    import kafka
    print("   kafka-python-ng ja instalado")
except ModuleNotFoundError:
    print("   Instalando kafka-python-ng...")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "kafka-python-ng", "--quiet"]
    )
    print("   kafka-python-ng instalado")

# -- 4. Kafka download + extracao --
print("-- 4/6  Kafka --")
start_sh = f"{BIN}/zookeeper-server-start.sh"

if os.path.isfile(start_sh):
    print(f"   Kafka ja instalado em {KAFKA_DIR}")
else:
    print(f"   Baixando Kafka {KAFKA_VERSION} (~113 MB)...")
    os.system(f"rm -rf {KAFKA_DIR} {KAFKA_ARCHIVE}")

    ret = os.system(f"wget -q --show-progress {KAFKA_URL} -O {KAFKA_ARCHIVE}")

    if ret != 0 or not os.path.isfile(KAFKA_ARCHIVE):
        raise RuntimeError("Download falhou. Verifique a conexao.")

    size_mb = os.path.getsize(KAFKA_ARCHIVE) / (1024 * 1024)
    print(f"   Arquivo baixado: {size_mb:.1f} MB")
    if size_mb < 50:
        raise RuntimeError(f"Arquivo muito pequeno ({size_mb:.1f} MB) — download incompleto.")

    print("   Extraindo...")
    os.system(f"tar -xzf {KAFKA_ARCHIVE} -C /opt/")

    if not os.path.isfile(start_sh):
        raise RuntimeError(f"Extracao falhou — {start_sh} nao encontrado.")

    print(f"   Kafka extraido com sucesso em {KAFKA_DIR}")

# -- 5. Configuracao do broker --
print("-- 5/6  Configuracao do Broker --")
os.system("pkill -f kafka.Kafka 2>/dev/null")
os.system("pkill -f zookeeper 2>/dev/null")
time.sleep(3)
os.system("rm -rf /tmp/kafka-logs /tmp/zookeeper")

server_props = f"{KAFKA_DIR}/config/server.properties"
os.system(f"sed -i '/^advertised\\.listeners/d' {server_props}")
os.system(f"sed -i '/^listeners=/d' {server_props}")
os.system(f"sed -i '/^log\\.dirs/d' {server_props}")
with open(server_props, "a") as f:
    f.write("\nlisteners=PLAINTEXT://0.0.0.0:9092\n")
    f.write("advertised.listeners=PLAINTEXT://localhost:9092\n")
    f.write("log.dirs=/tmp/kafka-logs\n")

check = subprocess.run(
    ["grep", "-E", "^listeners=|^advertised", server_props],
    capture_output=True, text=True
)
print(f"   {check.stdout.strip().replace(chr(10), ' | ')}")
print("   server.properties OK")

# -- 6. Servicos --
print("-- 6/6  Servicos --")

subprocess.Popen(
    f"{BIN}/zookeeper-server-start.sh {KAFKA_DIR}/config/zookeeper.properties".split(),
    stdout=open("/tmp/zookeeper.log", "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)
if not aguardar_porta(2181, "ZooKeeper"):
    os.system("tail -20 /tmp/zookeeper.log")
    raise RuntimeError("ZooKeeper nao subiu.")

subprocess.Popen(
    f"{BIN}/kafka-server-start.sh {KAFKA_DIR}/config/server.properties".split(),
    stdout=open("/tmp/kafka.log", "w"),
    stderr=subprocess.STDOUT,
    preexec_fn=os.setsid
)
if not aguardar_porta(9092, "Kafka Broker", max_seg=35):
    os.system("tail -20 /tmp/kafka.log")
    raise RuntimeError("Kafka Broker nao subiu.")

# -- Imports --
import json, threading, uuid, random
from datetime import datetime, timezone, timedelta
from collections import defaultdict, deque
from kafka import KafkaProducer, KafkaConsumer, KafkaAdminClient
from kafka.admin import NewTopic
from kafka.errors import TopicAlreadyExistsError

# -- Especifico Demo 3: topicos + helpers --
BOOTSTRAP = "localhost:9092"

TOPICOS = [
    NewTopic("transacoes-raw",      num_partitions=3, replication_factor=1),
    NewTopic("transacoes-enriched", num_partitions=3, replication_factor=1),
    NewTopic("alertas-fraude",      num_partitions=1, replication_factor=1),
    NewTopic("metricas-janela",     num_partitions=1, replication_factor=1),
]

admin = KafkaAdminClient(bootstrap_servers=BOOTSTRAP)
for t in TOPICOS:
    try:
        admin.create_topics([t])
        print(f"Topico criado: {t.name}")
    except TopicAlreadyExistsError:
        print(f"Topico ja existe: {t.name}")

def make_producer():
    return KafkaProducer(
        bootstrap_servers=BOOTSTRAP,
        value_serializer=lambda v: json.dumps(v, default=str).encode('utf-8'),
        key_serializer=lambda k: k.encode('utf-8') if k else None,
        acks='all',
    )

def make_consumer(topic, group_id, reset='latest'):
    return KafkaConsumer(
        topic,
        bootstrap_servers=BOOTSTRAP,
        group_id=group_id,
        auto_offset_reset=reset,
        enable_auto_commit=True,
        value_deserializer=lambda b: json.loads(b.decode('utf-8')),
        key_deserializer=lambda b: b.decode('utf-8') if b else None,
        consumer_timeout_ms=5000,
    )

print("\n" + "=" * 42)
print("Kafka pronto! Pode continuar.")
print("=" * 42)
print("\nPipeline configurado:")
print("   transacoes-raw -> [Transformador] -> transacoes-enriched")
print("                                     -> alertas-fraude")
print("   transacoes-enriched -> [Janela] -> metricas-janela")

---
## 🔄 Etapa 2 — Transformação de Eventos em Stream

In [ ]:
# ============================================================
# ETAPA 2.1 — GERADOR DE EVENTOS BRUTOS
# ============================================================
# Simula um sistema legado enviando transações no formato bruto.

CATEGORIAS_RISCO = {
    "eletronicos":   "ALTO",
    "joias":         "MUITO_ALTO",
    "roupas":        "BAIXO",
    "alimentos":     "BAIXO",
    "viagens":       "ALTO",
}

def gerar_transacao_raw():
    """Simula evento bruto de um sistema legado."""
    cat = random.choice(list(CATEGORIAS_RISCO.keys()))
    valor = round(random.uniform(5, 15000), 2)
    # Injeta transações suspeitas com 10% de chance
    if random.random() < 0.10:
        valor = round(random.uniform(8000, 50000), 2)  # Valor anômalo

    return {
        "txn_id":      str(uuid.uuid4())[:12],
        "usr":         f"{random.randint(1,500):04d}",  # user_id abreviado
        "cat":         cat,
        "val":         valor,                           # 'val' em vez de 'valor'
        "ts":          int(time.time() * 1000),         # Unix timestamp ms
        "src":         random.choice(["app", "web", "pos"]),
    }

#Utilize essa confguração no TS, caso queira acompanhar o fechamando diferente
#na etapa 3: ts=int(time.time() * 1000) - random.randint(0, 60000),

print(" Exemplo de evento RAW (legado):")
print(json.dumps(gerar_transacao_raw(), indent=2))

In [ ]:
# ============================================================
# ETAPA 2.2 — STREAM PROCESSOR: TRANSFORMAÇÃO + ENRIQUECIMENTO
# ============================================================
# Padrão: consume de 'transacoes-raw', transforma, produz em
# 'transacoes-enriched'. É o padrão Kafka Streams simplificado.

LIMITE_ALERTA = 5000.0   # R$ — threshold de alerta de fraude

def transformar_evento(raw):
    """Transforma evento bruto em evento enriquecido e padronizado."""
    ts_ms  = raw.get('ts', int(time.time() * 1000))
    ts_iso = datetime.fromtimestamp(ts_ms / 1000, tz=timezone.utc).isoformat()
    cat    = raw.get('cat', 'desconhecida')

    return {
        # Renomear campos para padrão
        "transaction_id":  raw['txn_id'],
        "user_id":         f"usr_{raw['usr']}",
        "categoria":       cat,
        "valor":           raw['val'],
        "canal":           raw.get('src', 'desconhecido'),
        # Enriquecer com dados calculados
        "nivel_risco":     CATEGORIAS_RISCO.get(cat, "MEDIO"),
        "suspeita_fraude": raw['val'] > LIMITE_ALERTA,
        "faixa_valor":     (
            "MICRO"  if raw['val'] < 50   else
            "PEQUENO" if raw['val'] < 500  else
            "MEDIO"  if raw['val'] < 5000 else "GRANDE"
        ),
        # Timestamps padronizados
        "event_time":      ts_iso,
        "process_time":    datetime.now(timezone.utc).isoformat(),
        "latencia_ms":     int(time.time() * 1000) - ts_ms,
    }

def stream_processor(n_eventos=20, verbose=True):
    """Processa eventos: raw → enriched, detecta fraudes."""
    prod_raw      = make_producer()
    prod_enriched = make_producer()
    prod_alertas  = make_producer()

    # 1. Produzir eventos brutos
    for _ in range(n_eventos):
        txn = gerar_transacao_raw()
        prod_raw.send("transacoes-raw", key=txn['usr'], value=txn)
    prod_raw.flush()
    if verbose:
        print(f"{n_eventos} eventos brutos enviados para 'transacoes-raw'")

    # 2. Consumir, transformar e re-publicar
    consumer = make_consumer("transacoes-raw", "processor-v1", reset='earliest')

    enriquecidos = 0
    alertas      = 0

    for msg in consumer:
        enriched = transformar_evento(msg.value)

        # Publicar evento enriquecido
        prod_enriched.send(
            "transacoes-enriched",
            key=enriched['user_id'],
            value=enriched
        )
        enriquecidos += 1

        # Publicar alerta se suspeito
        if enriched['suspeita_fraude']:
            alerta = {
                "tipo":           "FRAUDE_POTENCIAL",
                "transaction_id": enriched['transaction_id'],
                "user_id":        enriched['user_id'],
                "valor":          enriched['valor'],
                "nivel_risco":    enriched['nivel_risco'],
                "detectado_em":   datetime.now(timezone.utc).isoformat(),
            }
            prod_alertas.send("alertas-fraude", key=enriched['user_id'], value=alerta)
            alertas += 1
            if verbose:
                print(f"  ALERTA: {enriched['user_id']} | R$ {enriched['valor']:,.2f} | "
                      f"Risco: {enriched['nivel_risco']}")

    prod_enriched.flush()
    prod_alertas.flush()
    consumer.close()

    print(f"\n Processamento concluído:")
    print(f"   Enriquecidos : {enriquecidos}")
    print(f"   Alertas      : {alertas}")
    print(f"   Taxa suspeita: {alertas/enriquecidos*100:.1f}%" if enriquecidos else "")

stream_processor(n_eventos=30)

---
## 🪟 Etapa 3 — Janelas Tumbling (Fixas)

In [ ]:
# ============================================================
# ETAPA 3 — JANELA TUMBLING
# ============================================================
# Janela Tumbling: períodos fixos e NÃO sobrepostos.
# Ex: janela de 10s → [0-10s], [10-20s], [20-30s]...
#
# Uso típico: métricas por minuto, relatórios horários.

WINDOW_SEC = 10  # Janela de 10 segundos

class TumblingWindow:
    """Agrega eventos em janelas de tempo fixas."""

    def __init__(self, window_sec):
        self.window_sec = window_sec
        self.current_window_start = None
        self.buffer = []  # Eventos da janela atual
        self.closed_windows = []  # Janelas já fechadas

    def _window_start(self, ts):
        """Retorna o início da janela para um timestamp."""
        return (ts // self.window_sec) * self.window_sec

    def add(self, event, event_ts):
        """Adiciona evento. Retorna resultados da janela fechada, se houver."""
        win_start = self._window_start(event_ts)

        # Novo período → fechar janela anterior
        if self.current_window_start is not None and win_start != self.current_window_start:
            closed = self._close_window()
            self.buffer = []
            self.current_window_start = win_start
            self.buffer.append(event)
            return closed

        self.current_window_start = win_start
        self.buffer.append(event)
        return None

    def _close_window(self):
        if not self.buffer:
            return None
        valores = [e['valor'] for e in self.buffer]
        cats    = [e['categoria'] for e in self.buffer]
        result = {
            "window_start":  datetime.fromtimestamp(self.current_window_start, tz=timezone.utc).isoformat(),
            "window_end":    datetime.fromtimestamp(self.current_window_start + self.window_sec, tz=timezone.utc).isoformat(),
            "window_sec":    self.window_sec,
            "total_txns":    len(self.buffer),
            "valor_total":   round(sum(valores), 2),
            "valor_medio":   round(sum(valores) / len(valores), 2),
            "valor_max":     max(valores),
            "categoria_top": max(set(cats), key=cats.count),
            "suspeitas":     sum(1 for e in self.buffer if e.get('suspeita_fraude')),
        }
        self.closed_windows.append(result)
        return result

    def flush(self):
        """Força fechamento da janela corrente."""
        if self.buffer:
            return self._close_window()
        return None

print(" TumblingWindow definida — janela de", WINDOW_SEC, "segundos")

In [ ]:
# ============================================================
# EXECUTAR JANELA TUMBLING EM STREAM REAL
# ============================================================

prod_metrics = make_producer()
window = TumblingWindow(window_sec=WINDOW_SEC)
consumer_enr = make_consumer("transacoes-enriched", "window-processor", reset='earliest')

print(f" Processando eventos com janela tumbling de {WINDOW_SEC}s...\n")
fechadas = 0

for msg in consumer_enr:
    event = msg.value
    # Usar event_time do evento (não o tempo atual) — event-time processing
    try:
        ts = datetime.fromisoformat(event['event_time']).timestamp()
    except Exception:
        ts = time.time()

    result = window.add(event, ts)

    if result:
        fechadas += 1
        # Publicar métrica da janela
        prod_metrics.send("metricas-janela", value=result)
        print(f"     Janela fechada #{fechadas}:")
        print(f"     Período  : {result['window_start'][:19]} → {result['window_end'][:19]}")
        print(f"     Transações: {result['total_txns']:>4} | Total: R$ {result['valor_total']:>10,.2f} | "
              f"Médio: R$ {result['valor_medio']:>8,.2f}")
        print(f"     Top cat  : {result['categoria_top']} |  Suspeitas: {result['suspeitas']}")
        print()

# Fechar última janela
last = window.flush()
if last:
    prod_metrics.send("metricas-janela", value=last)
    print(f"  Última janela: {last['total_txns']} txns | R$ {last['valor_total']:,.2f}")

prod_metrics.flush()
consumer_enr.close()
print(f"\n {fechadas} janelas processadas")

---
## 🌊 Etapa 4 — Janelas Sliding (Deslizantes)

In [ ]:
# ============================================================
# ETAPA 4 — JANELA SLIDING
# ============================================================
# Janela Sliding: janela de tamanho fixo que avança continuamente.
# Ex: "últimos 60 segundos" recalculado a cada novo evento.
#
# Uso típico: detecção de anomalias, média móvel, rate limiting.

class SlidingWindow:
    """Mantém agregações para uma janela de tempo deslizante."""

    def __init__(self, window_sec):
        self.window_sec = window_sec
        # deque com (timestamp, event) — expira automaticamente
        self.events = deque()

    def add(self, event, ts):
        """Adiciona evento e remove os expirados."""
        self.events.append((ts, event))
        cutoff = ts - self.window_sec
        while self.events and self.events[0][0] < cutoff:
            self.events.popleft()

    def stats(self, ts_now):
        """Calcula estatísticas da janela atual."""
        if not self.events:
            return None
        valores = [e['valor'] for _, e in self.events]
        suspeitas = sum(1 for _, e in self.events if e.get('suspeita_fraude'))
        return {
            "window_end":       datetime.fromtimestamp(ts_now, tz=timezone.utc).isoformat(),
            "window_sec":       self.window_sec,
            "n_eventos":        len(valores),
            "valor_total":      round(sum(valores), 2),
            "valor_medio":      round(sum(valores) / len(valores), 2),
            "taxa_suspeitas":   round(suspeitas / len(valores) * 100, 1),
            "eventos_por_sec":  round(len(valores) / self.window_sec, 2),
        }

# Processar com janela deslizante de 30 segundos
sliding = SlidingWindow(window_sec=30)
consumer_slide = make_consumer("transacoes-enriched", "sliding-processor", reset='earliest')

print("Processando com janela SLIDING de 30 segundos...\n")
print(f"{'Janela até':<25} {'N eventos':>10} {'Total R$':>12} {'Médio R$':>10} {'Suspeitas%':>11} {'Evt/s':>6}")
print("-" * 80)

for msg in consumer_slide:
    event = msg.value
    try:
        ts = datetime.fromisoformat(event['event_time']).timestamp()
    except Exception:
        ts = time.time()

    sliding.add(event, ts)
    s = sliding.stats(ts)
    if s:
        print(f"  {s['window_end'][:22]:<23} {s['n_eventos']:>10} "
              f"  {s['valor_total']:>10,.0f} {s['valor_medio']:>10,.0f} "
              f"  {s['taxa_suspeitas']:>9.1f}% {s['eventos_por_sec']:>6.2f}")

consumer_slide.close()
print("\n Janela sliding concluída")

---
## 🚨 Etapa 5 — Sistema de Alertas

In [ ]:
# ============================================================
# ETAPA 5 — CONSUMIR ALERTAS DE FRAUDE
# ============================================================
# O tópico 'alertas-fraude' foi populado pelo stream_processor.
# Aqui simulamos um sistema de monitoramento que reage a eles.

consumer_alertas = make_consumer("alertas-fraude", "monitor-alertas", reset='earliest')

print(" MONITOR DE ALERTAS DE FRAUDE\n")
print(f"{'#':<4} {'User ID':<12} {'Valor R$':>12} {'Risco':>12} {'Detectado em':<28}")
print("-" * 72)

total_alertas   = 0
valor_em_risco  = 0.0
por_risco       = defaultdict(int)

for msg in consumer_alertas:
    alerta = msg.value
    total_alertas += 1
    valor_em_risco += alerta['valor']
    por_risco[alerta['nivel_risco']] += 1
    emoji = "🔴" if alerta['nivel_risco'] == "MUITO_ALTO" else "🟡"
    print(f"{emoji} {total_alertas:<3} {alerta['user_id']:<12} "
          f"  {alerta['valor']:>10,.2f} {alerta['nivel_risco']:>12} "
          f"  {alerta['detectado_em'][:26]}")

consumer_alertas.close()

print(f"\n{'='*72}")
print(f" RESUMO DOS ALERTAS:")
print(f"   Total de alertas    : {total_alertas}")
print(f"   Valor total em risco: R$ {valor_em_risco:,.2f}")
print(f"   Por nível de risco  :")
for risco, count in sorted(por_risco.items()):
    print(f"     {risco:<12}: {count}")

In [ ]:
# ============================================================
# ETAPA 5.2 — REGRA DE ALERTA POR FREQUÊNCIA
# ============================================================
# Padrão: usuário faz muitas transações em pouco tempo.
# Técnica: janela sliding por user_id (stateful per-key)

LIMITE_TXNS_POR_MINUTO = 5  # Alerta se usuário fizer > 5 txns/min

# Estado por usuário: deque de timestamps
user_windows = defaultdict(lambda: deque())

consumer_freq = make_consumer("transacoes-enriched", "freq-monitor", reset='earliest')
prod_alertas_freq = make_producer()

print(f" Monitorando frequência de transações (limite: {LIMITE_TXNS_POR_MINUTO} txns/min)...\n")

alertas_freq = 0

for msg in consumer_freq:
    event   = msg.value
    user_id = event['user_id']

    try:
        ts = datetime.fromisoformat(event['event_time']).timestamp()
    except Exception:
        ts = time.time()

    # Atualizar janela deslizante de 60s para este usuário
    dq = user_windows[user_id]
    dq.append(ts)
    cutoff = ts - 60
    while dq and dq[0] < cutoff:
        dq.popleft()

    # Verificar limite
    if len(dq) > LIMITE_TXNS_POR_MINUTO:
        alertas_freq += 1
        alerta = {
            "tipo":         "ALTA_FREQUENCIA",
            "user_id":      user_id,
            "txns_60s":     len(dq),
            "limite":       LIMITE_TXNS_POR_MINUTO,
            "detectado_em": datetime.now(timezone.utc).isoformat(),
        }
        prod_alertas_freq.send("alertas-fraude", key=user_id, value=alerta)
        print(f"   ALTA FREQUÊNCIA: {user_id} fez {len(dq)} transações em 60s!")

prod_alertas_freq.flush()
consumer_freq.close()
print(f"\n {alertas_freq} alertas de alta frequência detectados")